In [1]:
# !pip install transformers accelerate
from transformers import AutoTokenizer, AutoModel, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoConfig,Trainer, TrainingArguments,DataCollatorWithPadding
import torch
import numpy as np
import pandas as pd
# !pip install datasets
# from datasets import load_metric
# !pip install evaluate
# 
from sklearn.model_selection import train_test_split
import csv
# from optuna import Trial
from typing import Dict, Union, Any
import evaluate

2026-02-21 18:15:35.633275: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-21 18:15:35.684940: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-21 18:15:36.660333: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/root/miniconda3/lib

In [2]:
CSV_PATH = "data.with_changescribe.csv"

# =========================
# 2) 读 CSV -> Dataset -> split
# =========================
df = pd.read_csv(CSV_PATH)
label2id={'Adaptive':0, 'Corrective':1, 'Perfective':2}
df = df.replace({"labels": label2id})
df


,user,repo,commit,labels,msgs,diffs,feature,gt_clean,changescribe_text,core_diff
0,ponsonio,RxJava,0531b8bff5c14d9504beefb4ad47f473e3a22932,2,Change hasException to hasThrowable--,diff --git a/rxjava-core/src/main/java/rx/Noti...,"[1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Change hasException to hasThrowable,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Notifi...
1,ponsonio,RxJava,0950c46beda335819928585f1262dfe1dca78a0b,0,Trying to extend the Scheduler interface accor...,diff --git a/rxjava-core/src/main/java/rx/Sche...,"[2, 44, 0, 0, 30, 0, 0, 1, 18, 0, 0, 0, 0, 0, ...",Trying to extend the Scheduler interface accor...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Schedu...
2,ponsonio,RxJava,0f92fdd8e6422d5b79c610a7fd8409d222315a49,0,RunAsync method for outputting multiple values--,diff --git a/rxjava-contrib/rxjava-async-util/...,"[2, 53, 0, 0, 42, 0, 0, 1, 45, 1, 0, 0, 0, 0, ...",RunAsync method for outputting multiple values,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: util/async/Asy...
3,ponsonio,RxJava,100f571c9a2835d5a30a55374b9be74c147e031f,1,forEach with Action1 but not Observer--I re-re...,diff --git a/language-adaptors/rxjava-groovy/s...,"[1, 5, 122, 9, 10, 9, 4, 1, 5, 18, 2, 0, 0, 0,...",forEach with Action1 but not Observer I re-rea...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: lang/groovy/Ob...
4,ponsonio,RxJava,191f023cf5253ea90647bc091dcaf55ccdce81cc,1,1.x: Fix Completable swallows- OnErrorNotImple...,diff --git a/src/main/java/rx/Completable.java...,"[1, 1, 0, 0, 0, 0, 0, 1, 21, 0, 0, 0, 0, 0, 0,...",1.x: Completable swallows OnErrorNotImplemente...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Comple...
...,...,...,...,...,...,...,...,...,...,...
1776,jenkinsci,clearcase-plugin,51e9da224f80254476a7dc446bca817b505381d8,2,Use a temporary file to decrease memory consum...,diff --git a/src/main/java/hudson/plugins/clea...,"[2, 12, 0, 4, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,...",Use a temporary file to decrease memory consum...,clearcase-plugin [Change] ChangeScribeStart\nS...,Summarized Code Changes:\nFile: plugins/clearc...
1777,jexp,batch-import,609d6c4b1eea2c33d9fb950fcbb9ba9dc1f80fc3,2,added a more memory efficient structure for st...,diff --git a/src/main/java/org/neo4j/batchimpo...,"[10, 159, 29, 35, 9, 2, 1, 5, 106, 0, 4, 8, 0,...",added a more memory efficient structure for st...,batch-import [Change] ChangeScribeStart\nSumma...,Summarized Code Changes:\nFile: neo4j/batchimp...
1778,hdiv,hdiv,19b650c78a1c76f4fd90274d7f163f863c0d39e4,2,Memory and performance optimizations,diff --git a/hdiv-config/src/main/java/org/hdi...,"[31, 302, 131, 140, 170, 89, 53, 7, 88, 14, 17...",Memory and performance optimizations,hdiv [Change] ChangeScribeStart\nSummarized Co...,Summarized Code Changes:\nFile: config/xml/Con...
1779,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,2,Now it is possible to specify the sqlite open ...,diff --git a/pom.xml b/pom.xml\nindex 394263b....,"[5, 57, 20, 9, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Now it is possible to specify the sqlite open ...,persistence [Change] ChangeScribeStart\nSummar...,Summarized Code Changes:\nFile: pom.xml\n - Mo...


In [3]:
# df["diff_compact"] = df["diffs"].map(compress_diff_minimal)
# df["gen_prompt"] = df["diff_compact"].map(lambda s: "generate commit message: " + s)

In [4]:
# df["diff_compact"]

In [5]:
# from datasets import Dataset, load_metric

In [6]:
train, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp_df, test_size=0.5, random_state=42)

In [7]:
len(train)

1246

In [8]:
train['labels'].value_counts()

1    424
0    420
2    402
Name: labels, dtype: int64

In [9]:
test['labels'].value_counts()

2    100
1     89
0     79
Name: labels, dtype: int64

In [10]:
# id2label = {0: "Adaptive", 1: "POSITIVE"}
# label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    "/root/autodl-tmp/models/codebert-base", num_labels=3
)
tokenizer = AutoTokenizer.from_pretrained('/root/autodl-tmp/models/codebert-base')

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/models/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
training_args = TrainingArguments(
    output_dir="./my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none"
)

In [13]:
encoded_train = tokenizer(train['core_diff'].astype(str).to_list(), return_tensors='pt',truncation=True, padding=True)
encoded_test = tokenizer(test['core_diff'].astype(str).to_list(), return_tensors='pt',truncation=True, padding=True)
encoded_val = tokenizer(val['core_diff'].astype(str).to_list(), return_tensors='pt',truncation=True, padding=True)

In [14]:
class CommitDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [15]:
train_dataset = CommitDataset(encoded_train, list(train['labels']))
test_dataset = CommitDataset(encoded_test, list(test['labels']))
val_dataset = CommitDataset(encoded_val, list(val['labels']))

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy_score(labels, predictions)
    prec = precision_score(labels, predictions, average='weighted', zero_division=0)
    rec = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    # data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

/tmp/ipykernel_4683/3511894098.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.949208,0.546816,0.601536,0.546816,0.524016
2,No log,0.914250,0.558052,0.635421,0.558052,0.540699
3,No log,0.917202,0.614232,0.631715,0.614232,0.608456
4,No log,0.882114,0.647940,0.662138,0.647940,0.643839
5,No log,0.969622,0.632959,0.645847,0.632959,0.632819
6,No log,1.114904,0.621723,0.623737,0.621723,0.617990
7,0.691100,1.375474,0.588015,0.597232,0.588015,0.580197


/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a 

TrainOutput(global_step=546, training_loss=0.6563441028525104, metrics={'train_runtime': 164.123, 'train_samples_per_second': 75.919, 'train_steps_per_second': 4.753, 'total_flos': 2294875229423616.0, 'train_loss': 0.6563441028525104, 'epoch': 7.0})

In [18]:
fewshot_metrics = trainer.evaluate(eval_dataset=test_dataset)
fewshot_metrics

/tmp/ipykernel_4683/77294352.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


{'eval_loss': 0.9628849029541016,
 'eval_accuracy': 0.6119402985074627,
 'eval_precision': 0.6201354819702221,
 'eval_recall': 0.6119402985074627,
 'eval_f1': 0.612905587548081,
 'eval_runtime': 1.3497,
 'eval_samples_per_second': 198.566,
 'eval_steps_per_second': 12.596,
 'epoch': 7.0}

In [19]:
#发送多种类型的邮件
from email.mime.multipart import MIMEMultipart
import smtplib

from email.mime.text import MIMEText
msg_from = '915803745@qq.com'  # 发送方邮箱
passwd = 'vcuosuurrgkfbdai'   #就是上面的授权码
 
# to= ['g.zhang@gotion.com', 'j.tong@gotion.com'] #接受方邮箱
to= ['j.tong@gotion.com'] #接受方邮箱
#设置邮件内容
#MIMEMultipart类可以放任何内容
msg = MIMEMultipart()
conntent=f"{fewshot_metrics}"
#把内容加进去
msg.attach(MIMEText(conntent,'plain','utf-8'))
 
#设置邮件主题
msg['Subject']="强化学习模型训练完毕"
 
#发送方信息
msg['From']=msg_from
 
#开始发送
 
#通过SSL方式发送，服务器地址和端口
s = smtplib.SMTP_SSL("smtp.qq.com", 465)
# 登录邮箱
s.login(msg_from, passwd)
#开始发送
s.sendmail(msg_from,to,msg.as_string())
print("强化学习模型训练完毕")

强化学习模型训练完毕
